# 83 - Assignments dos experimentos selecionados no TCC II

**Objetivo:** reunir em uma única pasta os arquivos de atribuições (assignments) das nove configurações da Tabela 4 do TCC II (`docs/TCC_II_metodologia_resultados_revisao_2026-10-25.docx`), nomeados pelo experimento, e montar um Excel com uma aba por experimento para inspeção dos clusters (tabelas dinâmicas, filtros), como foi feito em `notebooks/reports/ClustersView.xlsx`.

**Origem dos dados:** execuções dos notebooks 70 a 76 de 24 e 25/10/2026, sobre a mesma amostra de 20.000 descrições (semente 42). Nenhum modelo é reajustado aqui: os arquivos são apenas copiados.

**Configurações incluídas (Tabela 4):** o primeiro colocado do ranking de cada combinação de representação e algoritmo, mais as candidatas A e B de TF-IDF + HDBSCAN escolhidas pelo autor para aprofundamento.

**Verificação:** antes de gravar, cada arquivo é conferido contra a Tabela 4 (20.000 linhas, quantidade de clusters sem contar ruído e quantidade de descrições em ruído). Qualquer divergência interrompe o notebook.

**Saídas:** `reports/tcc2_selecionados/` com os CSVs renomeados e `tcc2_experimentos_selecionados.xlsx`. Os CSVs já existentes e idênticos à origem são mantidos; um CSV com conteúdo diferente interrompe o notebook, a menos que `OVERWRITE = True`. O Excel é derivado dos CSVs e é sempre regravado.

In [1]:
from pathlib import Path
import filecmp
import logging
import shutil

import pandas as pd
from IPython.display import display
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook83")
pd.set_option("display.max_colwidth", 120)

## Configurações

Caminhos relativos à raiz do repositório. `EXPERIMENTS` registra, para cada configuração da Tabela 4, o arquivo de origem e os valores esperados de clusters e ruído. Silhueta e escore são copiados da Tabela 4 apenas como referência no resumo; eles não são recalculados aqui.

Observações sobre a origem:

- K-Means: o arquivo vem da pasta `review/` da execução, que guarda as atribuições das configurações revisadas.
- DBSCAN: `assignments.csv` da execução corresponde ao `selected_run_id` registrado em `run.json` (TF-IDF: `seed42_word_2_5_min_df10_eps0.025_min_samples3`; MPNet: `seed42_eps0.01_min_samples5`).
- TF-IDF + HDBSCAN primeiro do ranking: `min_samples` padrão equivale a `min_samples=5`, com os mesmos indicadores (subseção 5.4.1).

In [2]:
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei a raiz do repositório com data/processed/.")

OUTPUT_DIR = "reports/tcc2_selecionados"
EXCEL_NAME = "tcc2_experimentos_selecionados.xlsx"
OVERWRITE = False
EXPECTED_ROWS = 20_000
NOISE_LABEL = -1
CLUSTER_COLORS = ("DDEBF7", "B4C6E7")  # tons alternados entre clusters vizinhos
NOISE_COLOR = "EDEDED"

EXPERIMENTS = [
    {
        "id": "01", "aba": "01 TFIDF KMeans", "notebook": "70",
        "representacao": "TF-IDF", "algoritmo": "K-Means", "destaque": "1º do ranking",
        "parametros": "word (2,5), min_df=10, k=200",
        "nome": "01_tfidf_kmeans_word_2_5_min_df10_k200",
        "origem": "reports/tfidf_kmeans/202501-202512_20261024T183332Z_8ae789b8/review/word_2_5_min10_maxdf1_features100000_k200_seed42/assignments.csv",
        "clusters": 200, "ruido": 0, "silhueta": 0.2099, "escore": 0.7037,
    },
    {
        "id": "02", "aba": "02 MPNet KMeans", "notebook": "71",
        "representacao": "MPNet", "algoritmo": "K-Means", "destaque": "1º do ranking",
        "parametros": "k=200",
        "nome": "02_mpnet_kmeans_k200",
        "origem": "reports/mpnet_kmeans/202501-202512_20261024T211600Z_cb2c9e42/review/mpnet_k200_seed42/assignments.csv",
        "clusters": 200, "ruido": 0, "silhueta": 0.1165, "escore": 0.6687,
    },
    {
        "id": "03", "aba": "03 TFIDF DBSCAN", "notebook": "72",
        "representacao": "TF-IDF", "algoritmo": "DBSCAN", "destaque": "1º do ranking",
        "parametros": "word (2,5), min_df=10, eps=0.025, min_samples=3",
        "nome": "03_tfidf_dbscan_word_2_5_min_df10_eps0.025_min_samples3",
        "origem": "reports/tfidf_dbscan/202501-202512_20261025T031512Z159795/assignments.csv",
        "clusters": 734, "ruido": 12_783, "silhueta": 0.9996, "escore": 0.8401,
    },
    {
        "id": "04", "aba": "04 MPNet DBSCAN", "notebook": "73",
        "representacao": "MPNet", "algoritmo": "DBSCAN", "destaque": "1º do ranking",
        "parametros": "eps=0.01, min_samples=5",
        "nome": "04_mpnet_dbscan_eps0.01_min_samples5",
        "origem": "reports/mpnet_dbscan/202501-202512_20261025T041045Z274853/assignments.csv",
        "clusters": 11, "ruido": 19_867, "silhueta": 0.9753, "escore": 0.7424,
    },
    {
        "id": "05", "aba": "05 TFIDF HDBSCAN 1o", "notebook": "74",
        "representacao": "TF-IDF", "algoritmo": "HDBSCAN", "destaque": "1º do ranking",
        "parametros": "char_wb (5,10), min_df=10, min_cluster_size=5, min_samples padrão",
        "nome": "05_tfidf_hdbscan_1o_char_wb_5_10_min_df10_mcs5_min_samplesdefault",
        "origem": "reports/tfidf_hdbscan/202501-202512_20261025T055138Z888595/assignments_seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samplesdefault_eom.csv",
        "clusters": 736, "ruido": 9_817, "silhueta": 0.5189, "escore": 0.6969,
    },
    {
        "id": "06", "aba": "06 TFIDF HDBSCAN A", "notebook": "74",
        "representacao": "TF-IDF", "algoritmo": "HDBSCAN", "destaque": "Candidata A",
        "parametros": "word (2,5), min_df=10, min_cluster_size=20, min_samples padrão",
        "nome": "06_tfidf_hdbscan_candidata_A_word_2_5_min_df10_mcs20_min_samplesdefault",
        "origem": "reports/tfidf_hdbscan/202501-202512_20261025T055138Z/assignments_seed42_word_2_5_min_df10_min_cluster_size20_min_samplesdefault_eom.csv",
        "clusters": 101, "ruido": 15_999, "silhueta": 0.7127, "escore": 0.6923,
    },
    {
        "id": "07", "aba": "07 TFIDF HDBSCAN B", "notebook": "74",
        "representacao": "TF-IDF", "algoritmo": "HDBSCAN", "destaque": "Candidata B",
        "parametros": "char_wb (5,10), min_df=10, min_cluster_size=5, min_samples=10",
        "nome": "07_tfidf_hdbscan_candidata_B_char_wb_5_10_min_df10_mcs5_min_samples10",
        "origem": "reports/tfidf_hdbscan/202501-202512_20261025T055138Z/assignments_seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samples10_eom.csv",
        "clusters": 381, "ruido": 12_053, "silhueta": 0.5656, "escore": 0.6865,
    },
    {
        "id": "08", "aba": "08 MPNet HDBSCAN", "notebook": "75",
        "representacao": "MPNet", "algoritmo": "HDBSCAN", "destaque": "1º do ranking",
        "parametros": "min_cluster_size=5, min_samples=10",
        "nome": "08_mpnet_hdbscan_mcs5_min_samples10",
        "origem": "reports/mpnet_hdbscan/202501-202512_20261025T061828Z036879/assignments_seed42_mpnet_min_cluster_size5_min_samples10_eom.csv",
        "clusters": 123, "ruido": 16_619, "silhueta": 0.4710, "escore": 0.5939,
    },
    {
        "id": "09", "aba": "09 Hibrido HDBSCAN", "notebook": "76",
        "representacao": "TF-IDF + MPNet (alpha=0.2)", "algoritmo": "HDBSCAN", "destaque": "1º do ranking",
        "parametros": "word (2,5), min_df=5, alpha=0.2, min_cluster_size=5, min_samples=2",
        "nome": "09_tfidf_mpnet_hdbscan_alpha0.2_word_2_5_min_df5_mcs5_min_samples2",
        "origem": "reports/tfidf_mpnet_hdbscan/assignments_202501-202512_20261025T135551Z608058_seed42_word_2_5_min_df5_alpha0.2_min_cluster_size5_min_samples2_eom/assignments_seed42_word_2_5_min_df5_alpha0.2_min_cluster_size5_min_samples2_eom.csv",
        "clusters": 982, "ruido": 10_329, "silhueta": 0.3487, "escore": 0.6266,
    },
]

## Funções

In [3]:
def load_and_check(experiment):
    """Lê o CSV de origem e confere linhas, clusters e ruído com a Tabela 4."""
    path = PROJECT_ROOT / experiment["origem"]
    if not path.exists():
        raise FileNotFoundError(f"{experiment['id']}: arquivo de origem não encontrado: {path}")
    assignments = pd.read_csv(path)
    labels = assignments["cluster"]
    clusters = labels[labels.ne(NOISE_LABEL)].nunique()
    noise = int(labels.eq(NOISE_LABEL).sum())
    observed = (len(assignments), clusters, noise)
    expected = (EXPECTED_ROWS, experiment["clusters"], experiment["ruido"])
    if observed != expected:
        raise ValueError(
            f"{experiment['id']}: (linhas, clusters, ruído) observado {observed} != esperado {expected}"
        )
    logger.info("%s conferido: %d clusters, %d em ruído", experiment["id"], clusters, noise)
    return assignments


def summary_row(experiment, assignments, copied_path):
    """Monta a linha do resumo de um experimento."""
    labels = assignments["cluster"]
    noise = int(labels.eq(NOISE_LABEL).sum())
    sizes = labels[labels.ne(NOISE_LABEL)].value_counts()
    return {
        "aba": experiment["aba"],
        "notebook": experiment["notebook"],
        "representacao": experiment["representacao"],
        "algoritmo": experiment["algoritmo"],
        "destaque": experiment["destaque"],
        "parametros": experiment["parametros"],
        "descricoes": len(assignments),
        "clusters": int(labels[labels.ne(NOISE_LABEL)].nunique()),
        "ruido": noise,
        "ruido_pct": round(noise / len(assignments) * 100, 3),
        "cobertura_pct": round((len(assignments) - noise) / len(assignments) * 100, 3),
        "maior_cluster": int(sizes.max()),
        "mediana_tamanho_cluster": float(sizes.median()),
        "silhueta_tabela4": experiment["silhueta"],
        "escore_tabela4": experiment["escore"],
        "arquivo": copied_path.name,
        "origem": experiment["origem"],
    }


def sort_for_inspection(assignments):
    """Ordena por cluster (ruído no fim) e frequência decrescente."""
    is_noise = assignments["cluster"].eq(NOISE_LABEL)
    return (
        assignments.assign(_ruido=is_noise)
        .sort_values(["_ruido", "cluster", "frequencia"], ascending=[True, True, False])
        .drop(columns="_ruido")
        .reset_index(drop=True)
    )


def color_by_cluster(worksheet, frame):
    """Pinta as linhas alternando dois tons por cluster; o ruído fica cinza.

    Usa formatação condicional pela paridade do cluster, então a cor acompanha
    a linha mesmo depois de filtrar ou reordenar a tabela no Excel.
    """
    cluster_column = get_column_letter(frame.columns.get_loc("cluster") + 1)
    cells = f"A2:{get_column_letter(frame.shape[1])}{len(frame) + 1}"
    rules = [
        (f"${cluster_column}2={NOISE_LABEL}", NOISE_COLOR),
        (f"ISODD(${cluster_column}2)", CLUSTER_COLORS[1]),
        (f"ISEVEN(${cluster_column}2)", CLUSTER_COLORS[0]),
    ]
    for formula, color in rules:
        fill = PatternFill(start_color=color, end_color=color, fill_type="solid")
        worksheet.conditional_formatting.add(cells, FormulaRule(formula=[formula], fill=fill, stopIfTrue=True))


def format_as_table(worksheet, table_name, frame, row_stripes=True):
    """Formata a aba como Tabela do Excel, com cabeçalho fixo e larguras ajustadas."""
    last_cell = f"{get_column_letter(frame.shape[1])}{len(frame) + 1}"
    table = Table(displayName=table_name, ref=f"A1:{last_cell}")
    table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=row_stripes)
    worksheet.add_table(table)
    worksheet.freeze_panes = "A2"
    for index, column in enumerate(frame.columns, start=1):
        width = max(len(str(column)), frame[column].astype(str).str.len().quantile(0.95))
        worksheet.column_dimensions[get_column_letter(index)].width = min(max(width + 2, 10), 80)

## Execução

### 1. Carregar e conferir os assignments

In [4]:
loaded = {experiment["id"]: load_and_check(experiment) for experiment in EXPERIMENTS}

2026-10-26 10:32:59,910 INFO 01 conferido: 200 clusters, 0 em ruído


2026-10-26 10:32:59,929 INFO 02 conferido: 200 clusters, 0 em ruído


2026-10-26 10:32:59,943 INFO 03 conferido: 734 clusters, 12783 em ruído


2026-10-26 10:32:59,958 INFO 04 conferido: 11 clusters, 19867 em ruído


2026-10-26 10:32:59,971 INFO 05 conferido: 736 clusters, 9817 em ruído


2026-10-26 10:32:59,981 INFO 06 conferido: 101 clusters, 15999 em ruído


2026-10-26 10:32:59,998 INFO 07 conferido: 381 clusters, 12053 em ruído


2026-10-26 10:33:00,020 INFO 08 conferido: 123 clusters, 16619 em ruído


2026-10-26 10:33:00,026 INFO 09 conferido: 982 clusters, 10329 em ruído


### 2. Copiar os CSVs com nomes por experimento

**Grava arquivos** em `reports/tcc2_selecionados/`. A cópia é byte a byte (`shutil.copy2`), sem alterar o conteúdo; CSVs já existentes e idênticos não são copiados de novo.

In [5]:
output_dir = PROJECT_ROOT / OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
excel_path = output_dir / EXCEL_NAME
conflicts = [
    experiment["nome"] for experiment in EXPERIMENTS
    if (output_dir / f"{experiment['nome']}.csv").exists()
    and not filecmp.cmp(PROJECT_ROOT / experiment["origem"], output_dir / f"{experiment['nome']}.csv", shallow=False)
]
if conflicts and not OVERWRITE:
    raise FileExistsError(f"CSVs já existem com conteúdo diferente da origem; use OVERWRITE = True para regravar: {conflicts}")

summary_rows = []
for experiment in EXPERIMENTS:
    copied_path = output_dir / f"{experiment['nome']}.csv"
    if copied_path.exists() and experiment["nome"] not in conflicts:
        logger.info("Já existe idêntico à origem: %s", copied_path.name)
    else:
        shutil.copy2(PROJECT_ROOT / experiment["origem"], copied_path)
        logger.info("Copiado %s", copied_path.name)
    summary_rows.append(summary_row(experiment, loaded[experiment["id"]], copied_path))

summary = pd.DataFrame(summary_rows)
display(summary.drop(columns="origem"))

2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 01_tfidf_kmeans_word_2_5_min_df10_k200.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 02_mpnet_kmeans_k200.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 03_tfidf_dbscan_word_2_5_min_df10_eps0.025_min_samples3.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 04_mpnet_dbscan_eps0.01_min_samples5.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 05_tfidf_hdbscan_1o_char_wb_5_10_min_df10_mcs5_min_samplesdefault.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 06_tfidf_hdbscan_candidata_A_word_2_5_min_df10_mcs20_min_samplesdefault.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 07_tfidf_hdbscan_candidata_B_char_wb_5_10_min_df10_mcs5_min_samples10.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 08_mpnet_hdbscan_mcs5_min_samples10.csv


2026-10-26 10:33:00,054 INFO Já existe idêntico à origem: 09_tfidf_mpnet_hdbscan_alpha0.2_word_2_5_min_df5_mcs5_min_samples2.csv


,aba,notebook,representacao,algoritmo,destaque,parametros,descricoes,clusters,ruido,ruido_pct,cobertura_pct,maior_cluster,mediana_tamanho_cluster,silhueta_tabela4,escore_tabela4,arquivo
0,01 TFIDF KMeans,70,TF-IDF,K-Means,1º do ranking,"word (2,5), min_df=10, k=200",20000,200,0,0.000,100.000,13885,25.0,0.2099,0.7037,01_tfidf_kmeans_word_2_5_min_df10_k200.csv
1,02 MPNet KMeans,71,MPNet,K-Means,1º do ranking,k=200,20000,200,0,0.000,100.000,231,91.5,0.1165,0.6687,02_mpnet_kmeans_k200.csv
2,03 TFIDF DBSCAN,72,TF-IDF,DBSCAN,1º do ranking,"word (2,5), min_df=10, eps=0.025, min_samples=3",20000,734,12783,63.915,36.085,126,7.0,0.9996,0.8401,03_tfidf_dbscan_word_2_5_min_df10_eps0.025_min_samples3.csv
3,04 MPNet DBSCAN,73,MPNet,DBSCAN,1º do ranking,"eps=0.01, min_samples=5",20000,11,19867,99.335,0.665,60,7.0,0.9753,0.7424,04_mpnet_dbscan_eps0.01_min_samples5.csv
4,05 TFIDF HDBSCAN 1o,74,TF-IDF,HDBSCAN,1º do ranking,"char_wb (5,10), min_df=10, min_cluster_size=5, min_samples padrão",20000,736,9817,49.085,50.915,98,10.0,0.5189,0.6969,05_tfidf_hdbscan_1o_char_wb_5_10_min_df10_mcs5_min_samplesdefault.csv
5,06 TFIDF HDBSCAN A,74,TF-IDF,HDBSCAN,Candidata A,"word (2,5), min_df=10, min_cluster_size=20, min_samples padrão",20000,101,15999,79.995,20.005,337,31.0,0.7127,0.6923,06_tfidf_hdbscan_candidata_A_word_2_5_min_df10_mcs20_min_samplesdefault.csv
6,07 TFIDF HDBSCAN B,74,TF-IDF,HDBSCAN,Candidata B,"char_wb (5,10), min_df=10, min_cluster_size=5, min_samples=10",20000,381,12053,60.265,39.735,173,15.0,0.5656,0.6865,07_tfidf_hdbscan_candidata_B_char_wb_5_10_min_df10_mcs5_min_samples10.csv
7,08 MPNet HDBSCAN,75,MPNet,HDBSCAN,1º do ranking,"min_cluster_size=5, min_samples=10",20000,123,16619,83.095,16.905,209,17.0,0.4710,0.5939,08_mpnet_hdbscan_mcs5_min_samples10.csv
8,09 Hibrido HDBSCAN,76,TF-IDF + MPNet (alpha=0.2),HDBSCAN,1º do ranking,"word (2,5), min_df=5, alpha=0.2, min_cluster_size=5, min_samples=2",20000,982,10329,51.645,48.355,102,7.0,0.3487,0.6266,09_tfidf_mpnet_hdbscan_alpha0.2_word_2_5_min_df5_mcs5_min_samples2.csv


### 3. Gerar o Excel

**Grava** `tcc2_experimentos_selecionados.xlsx`. A primeira aba (`Resumo`) lista os experimentos. Cada aba seguinte contém um assignment ordenado por cluster (ruído `-1` no fim) e frequência decrescente, formatado como Tabela do Excel. As linhas de um mesmo cluster têm o mesmo tom, alternando entre dois azuis de um cluster para o seguinte, e o ruído fica cinza; a cor vem de formatação condicional pela paridade do cluster e acompanha a linha ao filtrar ou reordenar. Para reproduzir a visão do `ClustersView.xlsx`, crie em cada aba uma tabela dinâmica com `cluster` e `descricao_normalizada` em Linhas, `Soma de frequencia` em Valores e `ruido` no Filtro.

In [6]:
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Resumo", index=False)
    format_as_table(writer.sheets["Resumo"], "t00_resumo", summary)
    for experiment in EXPERIMENTS:
        frame = sort_for_inspection(loaded[experiment["id"]])
        frame.to_excel(writer, sheet_name=experiment["aba"], index=False)
        worksheet = writer.sheets[experiment["aba"]]
        format_as_table(worksheet, f"t{experiment['id']}_assignments", frame, row_stripes=False)
        color_by_cluster(worksheet, frame)
logger.info("Excel gravado em %s", excel_path)

2026-10-26 10:33:17,208 INFO Excel gravado em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tcc2_selecionados\tcc2_experimentos_selecionados.xlsx


### 4. Conferir o Excel gravado

Relê cada aba e confirma linhas, clusters e ruído.

In [7]:
for experiment in EXPERIMENTS:
    sheet = pd.read_excel(excel_path, sheet_name=experiment["aba"])
    labels = sheet["cluster"]
    observed = (len(sheet), labels[labels.ne(NOISE_LABEL)].nunique(), int(labels.eq(NOISE_LABEL).sum()))
    expected = (EXPECTED_ROWS, experiment["clusters"], experiment["ruido"])
    if observed != expected:
        raise ValueError(f"{experiment['aba']}: Excel {observed} != esperado {expected}")
print(sorted(path.name for path in output_dir.iterdir()))

['01_tfidf_kmeans_word_2_5_min_df10_k200.csv', '02_mpnet_kmeans_k200.csv', '03_tfidf_dbscan_word_2_5_min_df10_eps0.025_min_samples3.csv', '04_mpnet_dbscan_eps0.01_min_samples5.csv', '05_tfidf_hdbscan_1o_char_wb_5_10_min_df10_mcs5_min_samplesdefault.csv', '06_tfidf_hdbscan_candidata_A_word_2_5_min_df10_mcs20_min_samplesdefault.csv', '07_tfidf_hdbscan_candidata_B_char_wb_5_10_min_df10_mcs5_min_samples10.csv', '08_mpnet_hdbscan_mcs5_min_samples10.csv', '09_tfidf_mpnet_hdbscan_alpha0.2_word_2_5_min_df5_mcs5_min_samples2.csv', 'tcc2_experimentos_selecionados.xlsx']
